In [1]:
import numpy as np
import pandas as pd

import pickle

import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

#from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, precision_score, recall_score

In [2]:
data = {
    "review_id": [1,2,3,4,5,6,7,8,9,10],
    "text": [
        "I love this product it is amazing and works great",
        "This is the worst purchase I have ever made",
        "Absolutely fantastic quality and great value",
        "Terrible experience the product broke in two days",
        "Very happy with the purchase highly recommend",
        "Waste of money very disappointed",
        "Excellent performance and very easy to use",
        "Not worth the price poor quality",
        "Superb build quality and fantastic support",
        "Horrible customer service and bad experience"
    ],
    "label": [
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative",
        "positive",
        "negative"
    ]
}

df = pd.DataFrame(data)
df.head(10)

,review_id,text,label
0,1,I love this product it is amazing and works great,positive
1,2,This is the worst purchase I have ever made,negative
2,3,Absolutely fantastic quality and great value,positive
3,4,Terrible experience the product broke in two days,negative
4,5,Very happy with the purchase highly recommend,positive
5,6,Waste of money very disappointed,negative
6,7,Excellent performance and very easy to use,positive
7,8,Not worth the price poor quality,negative
8,9,Superb build quality and fantastic support,positive
9,10,Horrible customer service and bad experience,negative


# Task 1: Basic Text Preprocessing
1. Convert all text to lowercase.
2. Remove punctuation.
3. Remove stop words.
4. Tokenize the sentences.
5. Create a new column called clean_text.


In [3]:
# Select the Required Columns only
df = df[['text','label']]
df.head()

,text,label
0,I love this product it is amazing and works great,positive
1,This is the worst purchase I have ever made,negative
2,Absolutely fantastic quality and great value,positive
3,Terrible experience the product broke in two days,negative
4,Very happy with the purchase highly recommend,positive


In [4]:
# Function for Basic Text Cleaning
def processing_data(sentence):
    celan_text = []
    ps = PorterStemmer()
    sentence=sentence.lower() # Convert all text to lowercase
    sentence=re.sub('[^a-z]', ' ', sentence) # Remove punctuation
    words=sentence.split() # Tokenize the sentences
    words=[word for word in words if word not in stopwords.words('english')] # Remove stop words
    #words= [ps.stem(word) for word in words] # Stemming
    celan_text = " ".join(words) 
    return celan_text
    

In [5]:
# Create a new column called clean_text.
df['clean_text'] = df['text'].apply(processing_data)
df.head()


,text,label,clean_text
0,I love this product it is amazing and works great,positive,love product amazing works great
1,This is the worst purchase I have ever made,negative,worst purchase ever made
2,Absolutely fantastic quality and great value,positive,absolutely fantastic quality great value
3,Terrible experience the product broke in two days,negative,terrible experience product broke two days
4,Very happy with the purchase highly recommend,positive,happy purchase highly recommend


# Task 2: Bag of Words (BoW)
1. Apply ***CountVectorizer.***
2. Extract:
    - Vocabulary
    - Document-Term Matrix
3. Convert the matrix into a DataFrame.

***Questions:***
- How many unique words are there?
- Which word appears most frequently?


###  CountVecotrizer - Just counts the occurence (frequency of each word in all sentences)

### convert corpus sentence to corpus words (non duplicate)
#### get all possible words (i,e vocabulary) - absolutely, amazing, bad, broke, build, customer,days, disappointed, easy, ever, excellent, experience, fantastic, great, happy, highly, horrible, love, made, money, performance, poor, price, product, purchase, quality - these are the column names hemselves

In [6]:
# CounterVectorizer
cv = CountVectorizer()

bow = cv.fit_transform(df['clean_text'])

In [7]:
# Vocabulary total 38 unique words
cv.get_feature_names_out() # columns are sorted in the alphabetical order

array(['absolutely', 'amazing', 'bad', 'broke', 'build', 'customer',
       'days', 'disappointed', 'easy', 'ever', 'excellent', 'experience',
       'fantastic', 'great', 'happy', 'highly', 'horrible', 'love',
       'made', 'money', 'performance', 'poor', 'price', 'product',
       'purchase', 'quality', 'recommend', 'service', 'superb', 'support',
       'terrible', 'two', 'use', 'value', 'waste', 'works', 'worst',
       'worth'], dtype=object)

In [8]:
# How many unique words are there?
# 10 documents and 38 unique vocabulary words
bow.shape

(10, 38)

In [9]:
# Document- Term Metrix
bow.toarray()

array([[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0,
        0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
       [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
       [0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
        1

In [10]:
# Converting the metrix into a Dataframe
df2 = pd.DataFrame(bow.toarray(), columns=cv.get_feature_names_out())

In [11]:
df2.head(10)

,absolutely,amazing,bad,broke,build,customer,days,disappointed,easy,ever,...,superb,support,terrible,two,use,value,waste,works,worst,worth
0,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,1,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,0,0,0,1,0,0,1,0,0,0,...,0,0,1,1,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,1,0,0,0
6,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,1,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
8,0,0,0,0,1,0,0,0,0,0,...,1,1,0,0,0,0,0,0,0,0
9,0,0,1,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
# Which word appears most frequently?

#df2.sum().sort_values(ascending=True)

df2.sum().idxmax()

'quality'

# Task 3: N-grams
1. Generate:
    - Unigrams
    - Bigrams
    - Trigrams
2. Compare vocabulary size for:
    - ngram_range=(1,1)
    - ngram_range=(1,2)
    - ngram_range=(1,3)

***Questions:***
- Which bigram appears most frequently?
- How does vocabulary size change?


In [13]:
# Unigrams
cv_n1 = CountVectorizer(ngram_range=(1,1))

bow = cv_n1.fit_transform(df['clean_text'])

print('Unique Bag of words or Vocabulary: ',cv_n1.get_feature_names_out())

print('\nNumber of words: ',len(cv_n1.get_feature_names_out()))

Unique Bag of words or Vocabulary:  ['absolutely' 'amazing' 'bad' 'broke' 'build' 'customer' 'days'
 'disappointed' 'easy' 'ever' 'excellent' 'experience' 'fantastic' 'great'
 'happy' 'highly' 'horrible' 'love' 'made' 'money' 'performance' 'poor'
 'price' 'product' 'purchase' 'quality' 'recommend' 'service' 'superb'
 'support' 'terrible' 'two' 'use' 'value' 'waste' 'works' 'worst' 'worth']

Number of words:  38


In [14]:
#Unigrams+Bigrams
cv_n2=CountVectorizer(ngram_range=(1,2))

bow_bi = cv_n2.fit_transform(df['clean_text'])

print('Unique Bag of words or Vocabulary: ', cv_n2.get_feature_names_out())

print('\nNumbers of words: ', len(cv_n2.get_feature_names_out()))

Unique Bag of words or Vocabulary:  ['absolutely' 'absolutely fantastic' 'amazing' 'amazing works' 'bad'
 'bad experience' 'broke' 'broke two' 'build' 'build quality' 'customer'
 'customer service' 'days' 'disappointed' 'easy' 'easy use' 'ever'
 'ever made' 'excellent' 'excellent performance' 'experience'
 'experience product' 'fantastic' 'fantastic quality' 'fantastic support'
 'great' 'great value' 'happy' 'happy purchase' 'highly'
 'highly recommend' 'horrible' 'horrible customer' 'love' 'love product'
 'made' 'money' 'money disappointed' 'performance' 'performance easy'
 'poor' 'poor quality' 'price' 'price poor' 'product' 'product amazing'
 'product broke' 'purchase' 'purchase ever' 'purchase highly' 'quality'
 'quality fantastic' 'quality great' 'recommend' 'service' 'service bad'
 'superb' 'superb build' 'support' 'terrible' 'terrible experience' 'two'
 'two days' 'use' 'value' 'waste' 'waste money' 'works' 'works great'
 'worst' 'worst purchase' 'worth' 'worth price']

Numbers 

In [15]:
#Unigrams+Bigrams+Trigrams
cv_n3=CountVectorizer(ngram_range=(1,3))

bow_tri = cv_n3.fit_transform(df['clean_text'])

print('Unique Bag of words or Vocabulary: ', cv_n3.get_feature_names_out())

print('\nNumbers of words: ', len(cv_n3.get_feature_names_out()))

Unique Bag of words or Vocabulary:  ['absolutely' 'absolutely fantastic' 'absolutely fantastic quality'
 'amazing' 'amazing works' 'amazing works great' 'bad' 'bad experience'
 'broke' 'broke two' 'broke two days' 'build' 'build quality'
 'build quality fantastic' 'customer' 'customer service'
 'customer service bad' 'days' 'disappointed' 'easy' 'easy use' 'ever'
 'ever made' 'excellent' 'excellent performance'
 'excellent performance easy' 'experience' 'experience product'
 'experience product broke' 'fantastic' 'fantastic quality'
 'fantastic quality great' 'fantastic support' 'great' 'great value'
 'happy' 'happy purchase' 'happy purchase highly' 'highly'
 'highly recommend' 'horrible' 'horrible customer'
 'horrible customer service' 'love' 'love product' 'love product amazing'
 'made' 'money' 'money disappointed' 'performance' 'performance easy'
 'performance easy use' 'poor' 'poor quality' 'price' 'price poor'
 'price poor quality' 'product' 'product amazing' 'product amazing work

In [16]:
#Unigrams+Bigrams
cv_n4=CountVectorizer(ngram_range=(1,4))

bow = cv_n4.fit_transform(df['clean_text'])

print('Unique Bag of words or Vocabulary: ', cv_n4.get_feature_names_out())

print('\nNumbers of words: ', len(cv_n4.get_feature_names_out()))

Unique Bag of words or Vocabulary:  ['absolutely' 'absolutely fantastic' 'absolutely fantastic quality'
 'absolutely fantastic quality great' 'amazing' 'amazing works'
 'amazing works great' 'bad' 'bad experience' 'broke' 'broke two'
 'broke two days' 'build' 'build quality' 'build quality fantastic'
 'build quality fantastic support' 'customer' 'customer service'
 'customer service bad' 'customer service bad experience' 'days'
 'disappointed' 'easy' 'easy use' 'ever' 'ever made' 'excellent'
 'excellent performance' 'excellent performance easy'
 'excellent performance easy use' 'experience' 'experience product'
 'experience product broke' 'experience product broke two' 'fantastic'
 'fantastic quality' 'fantastic quality great'
 'fantastic quality great value' 'fantastic support' 'great' 'great value'
 'happy' 'happy purchase' 'happy purchase highly'
 'happy purchase highly recommend' 'highly' 'highly recommend' 'horrible'
 'horrible customer' 'horrible customer service'
 'horrible cust

In [17]:
# Comparing the Unigrams, Bigrams and Trigrams

print('Unigrams Number of words: ',len(cv_n1.get_feature_names_out()))
print('bigrams Number of words:  ',len(cv_n2.get_feature_names_out()))
print('trigrams Number of words: ',len(cv_n3.get_feature_names_out()))
print('trigrams Number of words: ',len(cv_n4.get_feature_names_out()))

Unigrams Number of words:  38
bigrams Number of words:   73
trigrams Number of words:  98
trigrams Number of words:  113


In [18]:
# Which bigram appears most frequently?

bi_df = pd.DataFrame(bow_bi.toarray(), columns=cv_n2.get_feature_names_out())


In [19]:
bi_grams = [w for w in cv_n2.get_feature_names_out() if len(w.split())==2]
bi_grams_freq = bi_df[bi_grams].sum().sort_values(ascending=True)
bi_grams_freq

absolutely fantastic     1
amazing works            1
bad experience           1
broke two                1
build quality            1
customer service         1
easy use                 1
ever made                1
excellent performance    1
experience product       1
fantastic quality        1
fantastic support        1
great value              1
happy purchase           1
highly recommend         1
horrible customer        1
love product             1
money disappointed       1
performance easy         1
poor quality             1
price poor               1
product amazing          1
product broke            1
purchase ever            1
purchase highly          1
quality fantastic        1
quality great            1
service bad              1
superb build             1
terrible experience      1
two days                 1
waste money              1
works great              1
worst purchase           1
worth price              1
dtype: int64

In [20]:
bi_grams_freq.idxmax()

'absolutely fantastic'

In [21]:
bi_grams_freq.max()

1

In [22]:
# How does vocabulary size change?


#Vocabulary Size increase when you increase the n_gram_range
#i.e.

print('Unigrams range (1,1) Number of words: ',len(cv_n1.get_feature_names_out()))
print('bigrams range (1,2) Number of words:  ',len(cv_n2.get_feature_names_out()))
print('trigrams range (1,3) Number of words: ',len(cv_n3.get_feature_names_out()))
print('FourGrams range (1,4) Number of words: ',len(cv_n4.get_feature_names_out()))

Unigrams range (1,1) Number of words:  38
bigrams range (1,2) Number of words:   73
trigrams range (1,3) Number of words:  98
FourGrams range (1,4) Number of words:  113


# Task 4: TF (Term Frequency)
1. Manually calculate TF for one document.
2. Compare it with CountVectorizer output.
   
***Formula:*** ***TF = Term Count / Total Terms in Document***

Meaning:

- Term Count → number of times a word appears
- Total Terms → total words in that document

In [23]:
#Manually calculate TF for one document.
tf =df['clean_text'][0]

words = tf.split()
tf_value = pd.Series(words).value_counts()/len(words)
print(tf_value.head())

love       0.2
product    0.2
amazing    0.2
works      0.2
great      0.2
Name: count, dtype: float64


In [24]:
# Compare it with CountVectorizer output.
cv = CountVectorizer()
bow = cv.fit_transform([tf])
# bow.toarray()
cv_bow = pd.DataFrame(bow.toarray(), columns=cv.get_feature_names_out())
# cv_bow.head()

cv_bow / cv_bow.sum(axis=1).values[0]

,amazing,great,love,product,works
0,0.2,0.2,0.2,0.2,0.2


### After convert CountVectorizer to TF values, both are same.

# Task 5: TF-IDF Vectorization
1. Apply TfidfVectorizer.
2. Convert output to DataFrame.
3. Identify top 5 highest TF-IDF words in:
    - One positive review
    - One negative review

***Questions:***
- Why do common words get lower TF-IDF scores?
- Which words are most important for classification?

In [25]:
# Apply TfidfVectorizer.
tf = TfidfVectorizer()
bow = tf.fit_transform(df['clean_text'])
# bow.toarray()
#tf.get_feature_names_out()

In [26]:
tf_df = pd.DataFrame(bow.toarray(), columns=tf.get_feature_names_out())
tf_df.head(10)

,absolutely,amazing,bad,broke,build,customer,days,disappointed,easy,ever,...,superb,support,terrible,two,use,value,waste,works,worst,worth
0,0.000000,0.474295,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.00000,0.474295,0.000000,0.000000
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.0,0.518291,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.00000,0.000000,0.518291,0.000000
2,0.500097,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0,0.500097,0.00000,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.000000,0.428537,0.000000,0.000000,0.428537,0.00000,0.0,0.000000,...,0.000000,0.000000,0.428537,0.428537,0.0,0.000000,0.00000,0.000000,0.000000,0.000000
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.00000,0.000000,0.000000,0.000000
5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.57735,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.57735,0.000000,0.000000,0.000000
6,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.5,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.5,0.000000,0.00000,0.000000,0.000000,0.000000
7,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.00000,0.000000,0.000000,0.530511
8,0.000000,0.000000,0.000000,0.000000,0.483606,0.000000,0.000000,0.00000,0.0,0.000000,...,0.483606,0.483606,0.000000,0.000000,0.0,0.000000,0.00000,0.000000,0.000000,0.000000
9,0.000000,0.000000,0.460158,0.000000,0.000000,0.460158,0.000000,0.00000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.00000,0.000000,0.000000,0.000000


In [27]:
df[['clean_text','label']]

,clean_text,label
0,love product amazing works great,positive
1,worst purchase ever made,negative
2,absolutely fantastic quality great value,positive
3,terrible experience product broke two days,negative
4,happy purchase highly recommend,positive
5,waste money disappointed,negative
6,excellent performance easy use,positive
7,worth price poor quality,negative
8,superb build quality fantastic support,positive
9,horrible customer service bad experience,negative


In [28]:
# Identify top 5 highest TF-IDF words in:
# One positive review
tf_df.iloc[0].sort_values(ascending=False).head(5)

amazing    0.474295
love       0.474295
works      0.474295
product    0.403194
great      0.403194
Name: 0, dtype: float64

In [29]:
# Identify top 5 highest TF-IDF words in:
# One negative review
tf_df.iloc[1].sort_values(ascending=False).head(5)

made          0.518291
ever          0.518291
worst         0.518291
purchase      0.440595
absolutely    0.000000
Name: 1, dtype: float64

In [ ]:
# Why do common words get lower TF-IDF scores?
# Because the IDF(Inverse Document Frequency) Penalizes words that apears in many Document

# TFIDF Formula:

# TF*IDF

# where:
# IDF = log(N/df)
# N-->total documents
# df--> number of documents containing the word
# If a word appears in many documents, then IDF becomes small, so TF-IDF becomes small.

In [ ]:
# Which words are most important for classification?

# Words with high TF-IDF scores.

# Examples:

# Positive sentiment words
# amazing    0.474295
# love       0.474295
# works      0.474295
# product    0.403194
# great      0.403194

# Negative sentiment words
# made          0.518291
# ever          0.518291
# worst         0.518291
# purchase      0.440595
# absolutely    0.000000

# These words help Machine Learning models classify sentiment.

# Task 6: Compare BoW vs TF-IDF
1. Train a Logistic Regression model using:
    - BoW features
    - TF-IDF features
2. Compare:
    - Accuracy
    - Precision
    - Recall
      
Which method performs better?

In [30]:
df['label']=df['label'].map({'positive':1,'negative':0})

In [31]:
df

,text,label,clean_text
0,I love this product it is amazing and works great,1,love product amazing works great
1,This is the worst purchase I have ever made,0,worst purchase ever made
2,Absolutely fantastic quality and great value,1,absolutely fantastic quality great value
3,Terrible experience the product broke in two days,0,terrible experience product broke two days
4,Very happy with the purchase highly recommend,1,happy purchase highly recommend
5,Waste of money very disappointed,0,waste money disappointed
6,Excellent performance and very easy to use,1,excellent performance easy use
7,Not worth the price poor quality,0,worth price poor quality
8,Superb build quality and fantastic support,1,superb build quality fantastic support
9,Horrible customer service and bad experience,0,horrible customer service bad experience


In [32]:
# Train a Logistic Regression model using:
# BoW features

x=df['clean_text']
y=df['label']

In [33]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [34]:
x_test

8    superb build quality fantastic support
1                  worst purchase ever made
Name: clean_text, dtype: object

In [35]:
cv=CountVectorizer()
x_train_bow = cv.fit_transform(x_train)
x_test_bow = cv.transform(x_test)

In [36]:
model_bow = LogisticRegression()

model_bow.fit(x_train_bow,y_train)
y_pred = model_bow.predict(x_test_bow)

In [37]:
y_pred

array([1, 1])

In [38]:
x_test_bow.toarray()

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

In [39]:
cv.get_feature_names_out()

array(['absolutely', 'amazing', 'bad', 'broke', 'customer', 'days',
       'disappointed', 'easy', 'excellent', 'experience', 'fantastic',
       'great', 'happy', 'highly', 'horrible', 'love', 'money',
       'performance', 'poor', 'price', 'product', 'purchase', 'quality',
       'recommend', 'service', 'terrible', 'two', 'use', 'value', 'waste',
       'works', 'worth'], dtype=object)

In [40]:
X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

cv = CountVectorizer()

# Fit only on training data
X_train_bow = cv.fit_transform(X_train)

# Transform test data using same vocabulary
X_test_bow = cv.transform(X_test)

model_bow = LogisticRegression()

model_bow.fit(X_train_bow, y_train)

y_pred = model_bow.predict(X_test_bow)

In [41]:
accuracy_bow = accuracy_score(y_test,y_pred)
precision_bow = precision_score(y_test,y_pred,pos_label=1)
recall_bow = recall_score(y_test,y_pred,pos_label=1)

print('Bow Accuracy: ',accuracy_bow)
print('Bow Precision: ',precision_bow)
print('Bow Recall: ',recall_bow)

Bow Accuracy:  0.5
Bow Precision:  0.5
Bow Recall:  1.0


In [ ]:
# Train a Logistic Regression model using:
# TF-IDF features

In [42]:
tfidf =TfidfVectorizer()

x_train_tf = tfidf.fit_transform(x_train)
x_test_tf = tfidf.transform(x_test)

model_tf = LogisticRegression()
model_tf.fit(x_train_tf,y_train)
pred_tf = model_tf.predict(x_test_tf)

accuracy_tf = accuracy_score(y_test, pred_tf)
precision_tf = precision_score(y_test, pred_tf)
recall_tf = recall_score(y_test, pred_tf)
print('TF of Accuracy: ',accuracy_tf)
print('TF Precision: ',precision_tf)
print('TF Recall: ',recall_tf)

TF of Accuracy:  0.5
TF Precision:  0.5
TF Recall:  1.0


In [125]:
# Which method performs better?

# Both are given the same scores

# Task 7: Stop Words Impact
1. Train model:
    - With stop words
    - Without stop words
2. Compare performance.

Does removing stop words improve accuracy?

In [43]:
# Without stop words
cv = CountVectorizer(
    stop_words = 'english'
)

x_train_bow_str = cv.fit_transform(x_train)
x_test_bow_str = cv.transform(x_test)

model_bow_str = LogisticRegression()
model_bow_str.fit(x_train_bow_str, y_train)
pred_bow_str = model_bow_str.predict(x_test_bow_str)
#pred_bow_str

accuracy_str = accuracy_score(y_test, pred_bow_str)
precision_str = precision_score(y_test, pred_bow_str)
recall_str = recall_score(y_test, pred_bow_str)
print('TF of Accuracy: ',accuracy_str)
print('TF Precision: ',precision_str)
print('TF Recall: ',recall_str)

TF of Accuracy:  0.5
TF Precision:  0.5
TF Recall:  1.0


In [44]:
# With stop words
cv = CountVectorizer()

x_train_bow_str = cv.fit_transform(x_train)
x_test_bow_str = cv.transform(x_test)

model_bow_str = LogisticRegression()
model_bow_str.fit(x_train_bow_str, y_train)
pred_bow_str = model_bow_str.predict(x_test_bow_str)
# pred_bow_str

accuracy_str = accuracy_score(y_test, pred_bow_str)
precision_str = precision_score(y_test, pred_bow_str)
recall_str = recall_score(y_test, pred_bow_str)
print('TF of Accuracy: ',accuracy_str)
print('TF Precision: ',precision_str)
print('TF Recall: ',recall_str)

TF of Accuracy:  0.5
TF Precision:  0.5
TF Recall:  1.0


In [45]:
# Comparison performance

print("WITH Stop Words")
print('TF of Accuracy: ',accuracy_str)
print('TF Precision: ',precision_str)
print('TF Recall: ',recall_str)

print("\nWITHOUT Stop Words")
print('TF of Accuracy: ',accuracy_str)
print('TF Precision: ',precision_str)
print('TF Recall: ',recall_str)

WITH Stop Words
TF of Accuracy:  0.5
TF Precision:  0.5
TF Recall:  1.0

WITHOUT Stop Words
TF of Accuracy:  0.5
TF Precision:  0.5
TF Recall:  1.0


In [ ]:
# Does removing stop words improve accuracy?

# usually after stop words removal will improve the model performance and metrics as well,
# but here bith given the same values due to less data we are doing the model traing and prediction

# Task 8: Text Classification (Full Pipeline)
***Build a full pipeline:***
- Preprocessing
- TF-IDF
- Train/Test Split
- Model Training
- Evaluation

***Use:***
- Logistic Regression
- Naive Bayes

Compare both models.

In [46]:
# Logistic Regression

# Preprocessing
def processing_data(sentence):
    celan_text = []
    ps = PorterStemmer()
    sentence=sentence.lower() # Convert all text to lowercase
    sentence=re.sub('[^a-z]', ' ', sentence) # Remove punctuation
    words=sentence.split() # Tokenize the sentences
    words=[word for word in words if word not in stopwords.words('english')] # Remove stop words
    #words= [ps.stem(word) for word in words] # Stemming
    celan_text = " ".join(words) 
    return celan_text
	
# Create a new column called clean_text.
df['clean_text'] = df['text'].apply(processing_data)
df.head()

# Train a Logistic Regression model using:
# BoW features

X = df['clean_text']
y = df['label']


# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# TF-IDF
tf = TfidfVectorizer()

# Fit only on training data
X_train_bow = tf.fit_transform(X_train)

# Transform test data using same vocabulary
X_test_bow = tf.transform(X_test)

model_bow = LogisticRegression()

# Model Training
model_bow.fit(X_train_bow, y_train)

y_pred = model_bow.predict(X_test_bow)

# Evaluation
accuracy_bow = accuracy_score(y_test,y_pred)
precision_bow = precision_score(y_test,y_pred,pos_label=1)
recall_bow = recall_score(y_test,y_pred,pos_label=1)

print('Bow Accuracy: ',accuracy_bow)
print('Bow Precision: ',precision_bow)
print('Bow Recall: ',recall_bow)


Bow Accuracy:  0.5
Bow Precision:  0.5
Bow Recall:  1.0


In [47]:
# Naive Bayes

nb_model = MultinomialNB()

# Model Training
nb_model.fit(X_train_bow, y_train)

y_pred = nb_model.predict(X_test_bow)

# Evaluation
accuracy_bow = accuracy_score(y_test,y_pred)
precision_bow = precision_score(y_test,y_pred,pos_label=1)
recall_bow = recall_score(y_test,y_pred,pos_label=1)

print('Bow Accuracy: ',accuracy_bow)
print('Bow Precision: ',precision_bow)
print('Bow Recall: ',recall_bow)

Bow Accuracy:  0.5
Bow Precision:  0.5
Bow Recall:  1.0


In [ ]:
# Compare both models.

# due to less data both model performing the same

# Task 9: Custom Stop Words
1. Add custom stop words like:
    ["product", "purchase"]
2. Retrain the model.
3. Analyze performance change.

In [48]:
custom_words = ["product", "purchase"]

In [49]:
# TF-IDF
tf = TfidfVectorizer(stop_words = custom_words)

# Fit only on training data
X_train_bow = tf.fit_transform(X_train)

# Transform test data using same vocabulary
X_test_bow = tf.transform(X_test)

model_bow = LogisticRegression()

# Model Training
model_bow.fit(X_train_bow, y_train)

y_pred = model_bow.predict(X_test_bow)

# Evaluation
accuracy_bow = accuracy_score(y_test,y_pred)
precision_bow = precision_score(y_test,y_pred,pos_label=1)
recall_bow = recall_score(y_test,y_pred,pos_label=1)

print('Bow Accuracy: ',accuracy_bow)
print('Bow Precision: ',precision_bow)
print('Bow Recall: ',recall_bow)


Bow Accuracy:  1.0
Bow Precision:  1.0
Bow Recall:  1.0


In [ ]:
# After Custom Stop words removal model given the 100% Accuracy

# Task 10: Feature Importance
1. Extract feature importance from Logistic Regression.
2. Identify:
    - Top 10 positive words
    - Top 10 negative words

Interpret results.

In [50]:
feature_names = tf.get_feature_names_out()

In [51]:
model_bow = LogisticRegression()

# Model Training
model_bow.fit(X_train_bow, y_train)
coefficients = model_bow.coef_[0]
coefficients

array([ 0.1916461 ,  0.20168622, -0.17957408, -0.17957408, -0.17957408,
       -0.17957408, -0.23132756,  0.20075783,  0.20075783, -0.30099419,
        0.1916461 ,  0.32964318,  0.23181518,  0.23181518, -0.17957408,
        0.20168622, -0.23132756,  0.20075783, -0.21527791, -0.21527791,
       -0.0198053 ,  0.23181518, -0.17957408, -0.17957408, -0.17957408,
        0.20075783,  0.1916461 , -0.23132756,  0.20168622, -0.21527791])

In [52]:
feature_importance = pd.DataFrame({
    'word': feature_names,
    'importance': coefficients
})

feature_importance.head(10)

,word,importance
0,absolutely,0.191646
1,amazing,0.201686
2,bad,-0.179574
3,broke,-0.179574
4,customer,-0.179574
5,days,-0.179574
6,disappointed,-0.231328
7,easy,0.200758
8,excellent,0.200758
9,experience,-0.300994


In [53]:
top_positive = feature_importance.sort_values(
    by='importance', ascending=False
).head(10)

top_positive

,word,importance
11,great,0.329643
12,happy,0.231815
13,highly,0.231815
21,recommend,0.231815
15,love,0.201686
1,amazing,0.201686
28,works,0.201686
17,performance,0.200758
7,easy,0.200758
25,use,0.200758


- great
- happy
- highly
- recommend
- love
- amazing
- works
- performance
- easy
- use

have high positive coefficients, meaning they strongly indicate ***positive sentiment.***

In [54]:
top_negative = feature_importance.sort_values(
    by='importance'
).head(10)

top_negative

,word,importance
9,experience,-0.300994
6,disappointed,-0.231328
27,waste,-0.231328
16,money,-0.231328
18,poor,-0.215278
19,price,-0.215278
29,worth,-0.215278
4,customer,-0.179574
23,terrible,-0.179574
22,service,-0.179574


- experience
- disappointed
- waste
- money
- poor
- price
- worth
- customer
- terrible
- service
have negative coefficients, meaning they strongly indicate ***negative sentiment.***

## Build a Sentiment Analyzer:
1. Train using TF-IDF.
2. Save the model using pickle.
3. Create a function:

    def predict_sentiment(text):
        # return positive or negative

   
### Test with new examples:
***"This product is fantastic"***

***"Worst quality ever"***


In [55]:
# 1. Train using TF-IDF.

# Create a new column called clean_text.
df['clean_text'] = df['text'].apply(processing_data)
df.head()

# Train a Logistic Regression model using:
# BoW features

X = df['clean_text']
y = df['label']


# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# TF-IDF
tf = TfidfVectorizer()

# Fit only on training data
X_train_bow = tf.fit_transform(X_train)

# Transform test data using same vocabulary
X_test_bow = tf.transform(X_test)

model_bow = LogisticRegression()

# Model Training
model_bow.fit(X_train_bow, y_train)


LogisticRegression()

In [56]:
# 2.Save the model using pickle.

pickle.dump(model_bow, open("sentiment_model.pkl", "wb"))
pickle.dump(tf, open("tfidf_vectorizer.pkl", "wb"))

In [57]:
model = pickle.load(open("sentiment_model.pkl", "rb"))
tfidf = pickle.load(open("tfidf_vectorizer.pkl", "rb"))

In [60]:
# 3.Create a function:
def predict_sentiment(text):

    text = processing_data(text)

    vector = tfidf.transform([text])

    pred = model.predict(vector)[0]

    if pred == 1:
        return "Positive"
    else:
        return "Negative"

In [61]:
print(predict_sentiment("This product is fantastic"))
print(predict_sentiment("Worst quality ever"))

Positive
Negative


# Conceptual Questions
1. Difference between CountVectorizer and TF-IDF?
2. Why is TF-IDF better for classification?
3. What happens if vocabulary size is too large?
4. What is sparsity in text data?
5. Why are n-grams useful?

### Answers:

1. Difference between CountVectorizer and TF-IDF?
| Aspect                | CountVectorizer (BoW)                | TF-IDF                           |
| --------------------- | ------------------------------------ | -------------------------------- |
| Concept               | Counts how many times a word appears | Weighs words based on importance |
| Output                | Raw frequency counts                 | Weighted frequency               |
| Formula               | Word Count                           | TF × IDF                         |
| Handling common words | Treats all words equally             | Reduces weight of common words   |
| Example               | "product" may get high count         | "product" may get low weight     |


2. Why is TF-IDF better for classification?

TF-IDF highlights important words and reduces the impact of frequent but less informative words.

Example:
Words like:

product, item, purchase

appear in many reviews.

TF-IDF assigns low importance to them, while giving higher weight to meaningful words:

Positive:

amazing, excellent, love

Negative:

bad, worst, broken

This helps the model learn better patterns, improving classification accuracy

3. What happens if vocabulary size is too large?

Large vocabulary causes several issues:

    1. High memory usage
    The feature matrix becomes very large.

    Example:
    10,000 documents
    50,000 unique words

    Matrix size:
    10000 × 50000
    2. Overfitting
    The model may learn noise instead of patterns.

    3. Slow training
    More features = more computation.

Solution
Use techniques like:
- Stop word removal
- Stemming / Lemmatization
- max_features
- min_df
- N-gram control
Example:
TfidfVectorizer(max_features=5000)

4. What is sparsity in text data?

Text data produces sparse matrices.

Sparse matrix = ***mostly zeros.***

Example:

Vocabulary:

love, product, bad, quality

Document:

love product

Matrix:

| love | product | bad | quality |
| ---- | ------- | --- | ------- |
| 1    | 1       | 0   | 0       |

Most values = 0.

This is called ***sparsity.***

Text datasets often have ***95%–99% zeros.***

Libraries like ***Scikit-learn use sparse matrices to save memory.***


5. Why are n-grams useful?

N-grams capture word relationships and context.

***Unigram (1 word)***
good
product

***Bigram (2 words)***
not good
very bad
customer service

***Trigram (3 words)***
not worth buying
very good product

***Why Important?***

Example:

Sentence:

This product is not good

Unigram model sees:

product, good

It might predict positive.

But bigram detects:

not good

Correctly predicting ***negative sentiment.***

| Concept          | Key Idea                                     |
| ---------------- | -------------------------------------------- |
| CountVectorizer  | Uses raw word counts                         |
| TF-IDF           | Weighs words by importance                   |
| Large vocabulary | Causes memory, speed, and overfitting issues |
| Sparsity         | Most feature values are zero                 |
| N-grams          | Capture word context and meaning             |


******************************End of the File******************************